In [1]:
# !pip install requests beautifulsoup4 tqdm pillow

import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm
from PIL import Image
from io import BytesIO

In [2]:
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

DATASET_DIR = "dataset/temporary"
os.makedirs(DATASET_DIR, exist_ok=True)

In [3]:
import re
from urllib.parse import urlparse, urljoin

def clean_url(url):
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

def get_domain_variants(netloc):
    netloc = netloc.lower()
    variants = [netloc]
    if netloc.startswith("www."):
        variants.append(netloc[4:])
    return variants

def extract_product_links(category_url, CONFIG=None):
    response = requests.get(category_url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    base_domain = urlparse(category_url).netloc.lower()

    config = None
    if CONFIG:
        for candidate in get_domain_variants(base_domain):
            if candidate in CONFIG:
                config = CONFIG[candidate]
                break

    if config:
        product_patterns = config.get("product_patterns", [])
        blacklist_patterns = config.get("blacklist", [])
    else:
        product_patterns = []
        blacklist_patterns = []

    links = set()

    for a in soup.find_all("a", href=True):
        href = urljoin(category_url, a["href"])
        href = clean_url(href)
        parsed = urlparse(href)

        # 1️⃣ même domaine uniquement
        if parsed.netloc.lower() not in get_domain_variants(base_domain):
            continue

        # 2️⃣ blacklist
        if any(re.search(pattern, href) for pattern in blacklist_patterns):
            continue

        # 3️⃣ whitelist stricte
        if any(re.search(pattern, href) for pattern in product_patterns):
            links.add(href)

    return list(links)

In [11]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from PIL import Image
from io import BytesIO

def extract_main_product_image(product_url, headers=None, min_size=400):
    response = requests.get(product_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    candidates = []

    for img in soup.find_all("img"):
        url = (
            img.get("data-src") or
            img.get("data-original") or
            img.get("srcset") or
            img.get("src")
        )

        if not url:
            continue

        # gérer srcset
        if "," in url:
            url = url.split(",")[-1].strip().split(" ")[0]

        url = urljoin(product_url, url)
        url = url.split("?")[0]

        if not url.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        if any(x in url.lower() for x in ["logo", "icon", "sprite"]):
            continue

        try:
            img_data = requests.get(url, headers=headers, timeout=5)
            image = Image.open(BytesIO(img_data.content)).convert("RGB")

            width, height = image.size

            if min(width, height) < min_size:
                continue

            # score = taille + bonus carré
            score = width * height

            ratio = width / height
            if 0.9 < ratio < 1.1:
                score *= 1.2  # bonus image carrée

            candidates.append((score, url))

        except:
            continue

    if not candidates:
        return None

    # retourner meilleure image
    candidates.sort(reverse=True)
    return candidates[0][1]

In [12]:
def download_image(url, save_folder, min_size=200):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")

        if min(img.size) < min_size:
            return False

        filename = os.path.basename(urlparse(url).path)
        save_path = os.path.join(save_folder, filename)

        img.save(save_path)
        return True

    except:
        return False

In [ ]:
def build_dataset(product_links):
    counter = 0

    for product_url in tqdm(product_links):
        try:
            image_url = extract_main_product_image(product_url)
            if image_url:
                if download_image(image_url, DATASET_DIR):
                    counter += 1
                else:
                    print("Failed to download:", image_url)

            time.sleep(0.1)  # anti-blocage

        except Exception as e:
            print("Error processing product URL:", product_url, e)
            continue

    print(f"{counter} images téléchargées")

In [7]:
SITE_CONFIG = {
    "ikea.com": {
        "product_patterns": [r"/p/.*-\d{8}/?$"],
        "blacklist": [r"/cat/", r"/f/", r"/customer-service/"]
    },
    "alinea.com": {
        "product_patterns": [r"/\d{8}/?$"],
        "blacklist": [r"/c/", r"/search"]
    }
}

category_pages = [
    "https://www.ikea.com/fr/fr/cat/chaises-700676/",
]

all_product_links = []

for page in category_pages:
    links = extract_product_links(page, CONFIG=SITE_CONFIG)
    all_product_links.extend(links)

all_product_links = list(set(all_product_links))

print(f"{len(all_product_links)} pages produit détectées")

38 pages produit détectées


In [8]:
#vérification rapide des liens
all_product_links[:5]

['https://www.ikea.com/fr/fr/p/krylbo-chaise-tonerud-bleu-90566744/',
 'https://www.ikea.com/fr/fr/p/odger-chaise-anthracite-50457313/',
 'https://www.ikea.com/fr/fr/p/skogsta-chaise-acacia-70544866/',
 'https://www.ikea.com/fr/fr/p/krylbo-chaise-tonerud-beige-fonce-60566745/',
 'https://www.ikea.com/fr/fr/p/teodores-chaise-noir-20530621/']

In [17]:
# Lancer le scrapping
build_dataset(all_product_links)

  3%|▎         | 1/38 [00:01<01:01,  1.67s/it]

Error processing product URL: https://www.ikea.com/fr/fr/p/krylbo-chaise-tonerud-bleu-90566744/ name 'image_urls' is not defined


  5%|▌         | 2/38 [00:03<00:56,  1.58s/it]

Error processing product URL: https://www.ikea.com/fr/fr/p/odger-chaise-anthracite-50457313/ name 'image_urls' is not defined


  8%|▊         | 3/38 [00:04<00:54,  1.56s/it]

Error processing product URL: https://www.ikea.com/fr/fr/p/skogsta-chaise-acacia-70544866/ name 'image_urls' is not defined


 11%|█         | 4/38 [00:06<00:51,  1.53s/it]

Error processing product URL: https://www.ikea.com/fr/fr/p/krylbo-chaise-tonerud-beige-fonce-60566745/ name 'image_urls' is not defined


 11%|█         | 4/38 [00:07<01:00,  1.78s/it]


KeyboardInterrupt: 

In [ ]:
import os

# Vider le dossier dataset temporaire

DATASET_DIR = "filtered_images"

for filename in os.listdir(DATASET_DIR):
    if "mirror" in filename.lower() or "enhanced" in filename.lower():
        file_path = os.path.join(DATASET_DIR, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)

In [ ]:
# Eliminer les doublons dans le dataset final
FINAL_DIR = "dataset/verified"

seen_hashes = set()

print("Before removing duplicates, number of images:", len(os.listdir(FINAL_DIR)))

for filename in os.listdir(FINAL_DIR):
    file_path = os.path.join(FINAL_DIR, filename)
    if os.path.isfile(file_path):
        try:
            img = Image.open(file_path)
            img_hash = hash(img.tobytes())
            if img_hash in seen_hashes:
                os.remove(file_path)
            else:
                seen_hashes.add(img_hash)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("After removing duplicates, number of images:", len(os.listdir(DATASET_DIR)))

Before removing duplicates, number of images: 0
After removing duplicates, number of images: 0


In [ ]:
# Transférer les images dans le dossier de sauvegarde final
FINAL_DIR = "dataset/verified"
os.makedirs(FINAL_DIR, exist_ok=True)
for filename in os.listdir(DATASET_DIR):
    src_path = os.path.join(DATASET_DIR, filename)
    dst_path = os.path.join(FINAL_DIR, filename)
    if os.path.isfile(src_path):
        os.rename(src_path, dst_path)

print(f"{len(os.listdir(FINAL_DIR))} images finales prêtes dans {FINAL_DIR}")